# Pipeline

> Run OCR + fix the markdown headings + describe images/figures for a single pdf file

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs, ocr_pdf
from mistocr.refine import add_img_descs, fix_hdgs
from pathlib import Path
from asyncio import Semaphore, gather, sleep
import os, json, shutil

In [ ]:
@delegates(add_img_descs)
async def pdf_to_md(
    pdf_path:str, # Path to input PDF file
    dst:str, # Destination directory for output markdown
    ocr_output:str=None, # Optional OCR output directory (defaults to pdf_path stem)
    model:str='claude-sonnet-4-5', # Model to use for heading fixes and image descriptions
    add_img_desc:bool=True, # Whether to add image descriptions
    progress:bool=True, # Whether to show progress messages
    **kwargs):
    "Convert PDF to markdown with OCR, fixed heading hierarchy, and optional image descriptions"
    n_steps = 3 if add_img_desc else 2
    if progress: print(f"Step 1/{n_steps}: Running OCR on {pdf_path}...")
    ocr_dirs = ocr_pdf(pdf_path, ocr_output or 'ocr_temp')
    ocr_dir = ocr_dirs[0]
    if progress: print(f"Step 2/{n_steps}: Fixing heading hierarchy...")
    fix_hdgs(ocr_dir, model=model)
    if add_img_desc:
        if progress: print(f"Step 3/{n_steps}: Adding image descriptions...")
        await add_img_descs(ocr_dir, dst=dst, model=model, progress=progress, **kwargs)
    elif dst and Path(dst) != ocr_dir: shutil.copytree(ocr_dir, dst, dirs_exist_ok=True)
    if progress: print("Done!")

In [ ]:
await pdf_to_md('files/test/attention-is-all-you-need.pdf', 'files/test/md_test')

Step 1/3: Running OCR on files/test/attention-is-all-you-need.pdf...


Mistral batch job status: QUEUED


Mistral batch job status: RUNNING


Step 2/3: Fixing heading hierarchy...


Step 3/3: Adding image descriptions...
Loading existing descriptions from ocr_temp/attention-is-all-you-need/img_descriptions.json
Adding descriptions to 15 pages...
Done! Enriched pages saved to files/test/md_test
Done!


In [ ]:
#| eval: false
!ls -R files/test/md_test

files/test/md_test:
page_1.md   page_12.md	page_15.md  page_4.md  page_7.md
page_10.md  page_13.md	page_2.md   page_5.md  page_8.md
page_11.md  page_14.md	page_3.md   page_6.md  page_9.md


In [ ]:
md = read_pgs('files/test/md_test', join=True)
print(md[5000:8000])


tions. In these models, the number of operations required to relate signals from two arbitrary input or output positions grows in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes it more difficult to learn dependencies between distant positions [12]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22].
End-to-end memory networks are based on a recurrent attenti

In [ ]:
def nb_to_md(path):
    """Convert a Jupyter notebook to markdown format"""
    import json
    
    with open(path, 'r') as f:
        nb = json.load(f)
    
    markdown_content = []
    
    for cell in nb['cells']:
        if cell['cell_type'] == 'markdown':
            markdown_content.append(''.join(cell['source']))
            markdown_content.append('\n---\n')
        elif cell['cell_type'] == 'code':
            markdown_content.append('```python')
            markdown_content.append(''.join(cell['source']))
            markdown_content.append('```\n')
    
    return '\n'.join(markdown_content)


In [ ]:
ocr_m = nb_to_md('00_core.ipynb')
refine_m = nb_to_md('01_refine.ipynb')


It looks like the images are not described or at least inserted in the doc, can you further investigate this error based on $`ocr_m` and $`refine_m` modules?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me investigate why images aren't being described or inserted. Let me check what's in the output directory:

```python
Path('files/test/md_test').ls()
```

Let me also check if there's an `img` folder and `img_descriptions.json`:

```python
img_dir = Path('files/test/md_test/img')
desc_file = Path('files/test/md_test/img_descriptions.json')
print(f"img dir exists: {img_dir.exists()}")
print(f"desc file exists: {desc_file.exists()}")
if img_dir.exists(): print(f"Images: {img_dir.ls()}")
if desc_file.exists(): print(f"Descriptions: {json.loads(desc_file.read_text())}")
```

In [ ]:
Path('files/test/md_test').ls()

(#15) [Path('files/test/md_test/page_8.md'),Path('files/test/md_test/page_14.md'),Path('files/test/md_test/page_7.md'),Path('files/test/md_test/page_6.md'),Path('files/test/md_test/page_15.md'),Path('files/test/md_test/page_11.md'),Path('files/test/md_test/page_2.md'),Path('files/test/md_test/page_9.md'),Path('files/test/md_test/page_12.md'),Path('files/test/md_test/page_10.md'),Path('files/test/md_test/page_13.md'),Path('files/test/md_test/page_4.md'),Path('files/test/md_test/page_1.md'),Path('files/test/md_test/page_5.md'),Path('files/test/md_test/page_3.md')]

In [ ]:
img_dir = Path('files/test/md_test/img')
desc_file = Path('files/test/md_test/img_descriptions.json')
print(f"img dir exists: {img_dir.exists()}")
print(f"desc file exists: {desc_file.exists()}")
if img_dir.exists(): print(f"Images: {img_dir.ls()}")
if desc_file.exists(): print(f"Descriptions: {json.loads(desc_file.read_text())}")

img dir exists: False
desc file exists: False


What's you diagnostic?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The issue is clear: the `img` folder and image descriptions aren't being copied to the destination. Looking at the `pdf_to_md` function, when `add_img_desc=True`, it calls `add_img_descs(ocr_dir, dst=dst, ...)` but the `add_img_descs` function in the refine module doesn't copy the `img` folder to the destination.

Let me check what's in the OCR output directory:

```python
ocr_dir = Path('ocr_temp/attention-is-all-you-need')
print(f"OCR dir exists: {ocr_dir.exists()}")
if ocr_dir.exists(): print(f"Contents: {ocr_dir.ls()}")
img_dir_ocr = ocr_dir/'img'
if img_dir_ocr.exists(): print(f"Images in OCR: {img_dir_ocr.ls()[:5]}")
```

In [ ]:
ocr_dir = Path('ocr_temp/attention-is-all-you-need')
print(f"OCR dir exists: {ocr_dir.exists()}")
if ocr_dir.exists(): print(f"Contents: {ocr_dir.ls()}")
img_dir_ocr = ocr_dir/'img'
if img_dir_ocr.exists(): print(f"Images in OCR: {img_dir_ocr.ls()[:5]}")

OCR dir exists: True
Contents: [Path('ocr_temp/attention-is-all-you-need/page_8.md'), Path('ocr_temp/attention-is-all-you-need/page_14.md'), Path('ocr_temp/attention-is-all-you-need/page_7.md'), Path('ocr_temp/attention-is-all-you-need/page_6.md'), Path('ocr_temp/attention-is-all-you-need/page_15.md'), Path('ocr_temp/attention-is-all-you-need/page_11.md'), Path('ocr_temp/attention-is-all-you-need/page_2.md'), Path('ocr_temp/attention-is-all-you-need/page_9.md'), Path('ocr_temp/attention-is-all-you-need/img_descriptions.json'), Path('ocr_temp/attention-is-all-you-need/img'), Path('ocr_temp/attention-is-all-you-need/page_12.md'), Path('ocr_temp/attention-is-all-you-need/page_10.md'), Path('ocr_temp/attention-is-all-you-need/page_13.md'), Path('ocr_temp/attention-is-all-you-need/page_4.md'), Path('ocr_temp/attention-is-all-you-need/page_1.md'), Path('ocr_temp/attention-is-all-you-need/page_5.md'), Path('ocr_temp/attention-is-all-you-need/page_3.md')]
Images in OCR: [Path('ocr_temp/attenti

What do you think?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The problem is in `add_img_descs` - it only copies the `img` folder when `dst_path != src_path` AND when the source images exist. But in your pipeline, you're calling it with `dst=dst` while the function is checking `src_path` for images. The images are in `ocr_dir` but the function never copies them to `dst`.

Looking at the refine module, `add_img_descs` has this logic:
```python
src_path,dst_path = Path(src),Path(dst) if dst else Path(src)
if dst_path != src_path: dst_path.mkdir(parents=True, exist_ok=True)
```

But there's no code to copy the `img` folder from `src_path` to `dst_path`. Let me check if a page has image references:

```python
pg = Path('ocr_temp/attention-is-all-you-need/page_3.md').read_text()
print(pg[2000:3000])
```

In [ ]:
src_path,dst_path = Path(src),Path(dst) if dst else Path(src)
if dst_path != src_path: dst_path.mkdir(parents=True, exist_ok=True)

NameError: name 'src' is not defined

In [ ]:
pg = Path('ocr_temp/attention-is-all-you-need/page_3.md').read_text()
print(pg[2000:3000])

What's missing for your investigation?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check if the page has image references and descriptions:

```python
pg = Path('ocr_temp/attention-is-all-you-need/page_3.md').read_text()
print(pg[2000:3500])
```

Also let me check the destination page to see if descriptions were added:

```python
pg_dst = Path('files/test/md_test/page_3.md').read_text()
print(pg_dst[2000:3500])
```

In [ ]:
pg = Path('ocr_temp/attention-is-all-you-need/page_3.md').read_text()
print(pg[:500])

![img-0.jpeg](img-0.jpeg)

Figure 1: The Transformer - model architecture.

The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.

### 3.1 Encoder and Decoder Stacks  ... page 3

Encoder: The encoder is composed of a stack of $N=6$ identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the


In [ ]:
pg_dst = Path('files/test/md_test/page_3.md').read_text()
print(pg_dst[:500])

![img-0.jpeg](img-0.jpeg)

Figure 1: The Transformer - model architecture.

The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.

### 3.1 Encoder and Decoder Stacks  ... page 3

Encoder: The encoder is composed of a stack of $N=6$ identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the


So?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The descriptions weren't added! The source and destination pages are identical. The issue is that `add_img_descs` loaded existing descriptions from the cache file but didn't actually insert them into the markdown. Let me check if the descriptions exist:

```python
desc_file = Path('ocr_temp/attention-is-all-you-need/img_descriptions.json')
descs = json.loads(desc_file.read_text())
print(f"Number of descriptions: {len(descs)}")
print(f"First description: {list(descs.items())[0]}")
```

In [ ]:
desc_file = Path('ocr_temp/attention-is-all-you-need/img_descriptions.json')
descs = json.loads(desc_file.read_text())
print(f"Number of descriptions: {len(descs)}")
print(f"First description: {list(descs.items())[0]}")

Number of descriptions: 5
First description: ('img-0.jpeg', {'is_informative': True, 'description': 'This is an architectural diagram of a Transformer model, showing both the encoder (left) and decoder (right) components. \n\nThe encoder (left side) processes inputs through:\n- Input Embedding layer at the bottom\n- Positional Encoding (added via ⊕ symbol)\n- N× stacked blocks, each containing:\n  - Masked Multi-Head Attention (orange)\n  - Add & Norm layer (yellow)\n  - Feed Forward layer (blue)\n  - Another Add & Norm layer (yellow)\n- Residual connections shown as curved arrows around each sub-layer\n\nThe decoder (right side) processes outputs (shifted right) through:\n- Output Embedding layer\n- Positional Encoding (added via ⊕ symbol)\n- N× stacked blocks, each containing:\n  - Masked Multi-Head Attention (orange)\n  - Add & Norm layer (yellow)\n  - Multi-Head Attention that receives input from encoder (orange)\n  - Add & Norm layer (yellow)\n  - Feed Forward layer (blue)\n  - Ad

Yes they do right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, the descriptions exist and are detailed! The problem is that `add_img_descs` isn't actually inserting them into the markdown. Looking at the refine module, the issue is in how `add_descs_to_pg` finds image references. Let me check what the actual image markdown looks like:

```python
import re
pg = Path('ocr_temp/attention-is-all-you-need/page_3.md').read_text()
img_links = re.findall(r'\[\^\d+\]!\[[^\]]*\]\([^)]+\)', pg)
print(f"Found {len(img_links)} image links with pattern")
print(f"Image links: {img_links}")
simple_imgs = re.findall(r'!\[[^\]]*\]\([^)]+\)', pg)
print(f"Found {len(simple_imgs)} simple image links")
print(f"Simple images: {simple_imgs[:3]}")
```

In [ ]:
import re
pg = Path('ocr_temp/attention-is-all-you-need/page_3.md').read_text()
img_links = re.findall(r'\[\^\d+\]!\[[^\]]*\]\([^)]+\)', pg)
print(f"Found {len(img_links)} image links with pattern")
print(f"Image links: {img_links}")
simple_imgs = re.findall(r'!\[[^\]]*\]\([^)]+\)', pg)
print(f"Found {len(simple_imgs)} simple image links")
print(f"Simple images: {simple_imgs[:3]}")

Found 0 image links with pattern
Image links: []
Found 1 simple image links
Simple images: ['![img-0.jpeg](img-0.jpeg)']


Let's move on?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Found it! The regex in `add_descs_to_pg` is looking for `\[\^\d+\]!\[[^\]]*\]\([^)]+\)` (with footnote references) but the actual markdown is just `![img-0.jpeg](img-0.jpeg)` (simple format). Let's fix the `add_descs_to_pg` function:

```python
def add_descs_to_pg(pg, descs):
    "Add AI-generated descriptions to images in page"
    for link in re.findall(r'!\[[^\]]*\]\([^)]+\)', pg):
        fname = re.findall(r'\(([^)]+)\)', link)[0]
        if fname in descs: pg = pg.replace(link, f"{link}\nAI-generated image description:\n___\n{descs[fname]['description']}\n___")
    return pg
```

In [ ]:
def add_descs_to_pg(pg, descs):
    "Add AI-generated descriptions to images in page"
    for link in re.findall(r'!\[[^\]]*\]\([^)]+\)', pg):
        fname = re.findall(r'\(([^)]+)\)', link)[0]
        if fname in descs: pg = pg.replace(link, f"{link}\nAI-generated image description:\n___\n{descs[fname]['description']}\n___")
    return pg

How to check?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let's test it with the actual page and descriptions:

```python
pg = Path('ocr_temp/attention-is-all-you-need/page_3.md').read_text()
descs = json.loads(Path('ocr_temp/attention-is-all-you-need/img_descriptions.json').read_text())
pg_enriched = add_descs_to_pg(pg, descs)
print(pg_enriched[:1500])
```

In [ ]:
pg = Path('ocr_temp/attention-is-all-you-need/page_3.md').read_text()
descs = json.loads(Path('ocr_temp/attention-is-all-you-need/img_descriptions.json').read_text())
pg_enriched = add_descs_to_pg(pg, descs)
print(pg_enriched[:1500])

![img-0.jpeg](img-0.jpeg)
AI-generated image description:
___
This is an architectural diagram of a Transformer model, showing both the encoder (left) and decoder (right) components. 

The encoder (left side) processes inputs through:
- Input Embedding layer at the bottom
- Positional Encoding (added via ⊕ symbol)
- N× stacked blocks, each containing:
  - Masked Multi-Head Attention (orange)
  - Add & Norm layer (yellow)
  - Feed Forward layer (blue)
  - Another Add & Norm layer (yellow)
- Residual connections shown as curved arrows around each sub-layer

The decoder (right side) processes outputs (shifted right) through:
- Output Embedding layer
- Positional Encoding (added via ⊕ symbol)
- N× stacked blocks, each containing:
  - Masked Multi-Head Attention (orange)
  - Add & Norm layer (yellow)
  - Multi-Head Attention that receives input from encoder (orange)
  - Add & Norm layer (yellow)
  - Feed Forward layer (blue)
  - Add & Norm layer (yellow)
- Residual connections around each s

So?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect! The fix works. Now we need to update the actual function in the refine module and also make sure the `img` folder gets copied to the destination. Let's create a patched version of `add_img_descs`:

```python
async def add_img_descs_fixed(src, dst=None, model='claude-sonnet-4-5', img_folder='img', semaphore=2, delay=1, force=False, progress=True):
    "Describe all images in markdown document and insert descriptions inline"
    src_path,dst_path = Path(src),Path(dst) if dst else Path(src)
    if dst_path != src_path: dst_path.mkdir(parents=True, exist_ok=True)
    src_imgs = src_path/img_folder
    if src_imgs.exists() and dst_path != src_path: shutil.copytree(src_imgs, dst_path/img_folder, dirs_exist_ok=True)
    desc_file = src_path/'img_descriptions.json'
    if desc_file.exists() and not force:
        if progress: print(f"Loading existing descriptions from {desc_file}")
        descs = json.loads(desc_file.read_text())
    else:
        imgs = (src_path/img_folder).ls(file_exts=['.jpeg', '.jpg', '.png'])
        if progress: print(f"Describing {len(imgs)} images...")
        descs = await describe_imgs(imgs, model, semaphore=semaphore, delay=delay)
        save_img_descs(descs, desc_file)
        if progress: print(f"Saved descriptions to {desc_file}")
    pgs = read_pgs(src_path, join=False)
    if progress: print(f"Adding descriptions to {len(pgs)} pages...")
    enriched = [add_descs_to_pg(pg, descs) for pg in pgs]
    for i,pg in enumerate(enriched, 1): (dst_path/f'page_{i}.md').write_text(pg)
    if progress: print(f"Done! Enriched pages saved to {dst_path}")
```